# **Proyecto: Transfer Learning para Clasificación de Pokémon (166 Clases)**
**Objetivo del Proyecto:**
Implementar transferencia de aprendizaje (*Transfer Learning*) y ajuste fino (*Fine-Tuning*) utilizando la red neuronal convolucional **MobileNetV2** preentrenada en ImageNet para clasificar imágenes de Pokémon de la Primera Generación y sus formas Mega (166 clases, >40,000 imágenes).

El pipeline incluye:
1. **Detección robusta y limpieza de datos:** Localización automática de las carpetas `train` y `test` (resolviendo la anidación del dataset en Kaggle) y filtrado de archivos no compatibles.
2. **Exploración de datos (EDA):** Distribución de imágenes por clase, visualización en cuadrícula y balanceo.
3. **Transfer Learning con MobileNetV2:**
   - **Fase 1:** Extracción de características con la base convolucional congelada.
   - **Fase 2:** Ajuste fino (*Fine-Tuning*) descongelando capas a partir de la capa 100 con learning rate reducido.
4. **Evaluación:** Curvas completas de Accuracy/Loss, evaluación en Test y Matriz de Confusión (`sns.heatmap`).
5. **Exportación para Netlify y App Móvil:** Conversión a **TensorFlow.js (`model.json` y fragmentos binarios `.bin`)** junto con `class_names.json` y empaquetado ZIP.


## **0. Instalación de Dependencias**
Instalamos las librerías necesarias en el entorno (Google Colab / Jupyter).


In [ ]:
# Instalación de dependencias
!pip install -q kagglehub tensorflowjs scikit-learn seaborn pillow pandas matplotlib


## **1. Importación de Librerías y Verificación de Hardware**
Importamos los módulos necesarios y comprobamos que la GPU esté disponible.


In [ ]:
import os
import sys
import json
import time
import shutil
from pathlib import Path

# Medición de tiempo
tiempo_inicial = time.time()
print("Tiempo de inicio registrado.")

# Librerías científicas y de visualización
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Machine Learning y Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix

print(f"Versión de TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU(s) detectada(s) ({len(gpus)}): {gpus}")
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as e:
            print("Configuración de memoria dinámica:", e)
else:
    print("AVISO: No se detectó GPU activa. Si estás en Google Colab, ve a Entorno de ejecución > Cambiar tipo de entorno y selecciona GPU (T4, L4 o A100).")


## **2. Descarga del Dataset y Resolución Automática de Rutas**
Descargamos el dataset oficial `mornville/pokemonimagedataset` mediante `kagglehub`.
Implementamos una detección recursiva que localiza con precisión las carpetas `train` y `test` dentro del directorio extraído (resolviendo cualquier subdirectorio anidado como `dataset/`).


In [ ]:
import kagglehub

# Descarga del dataset
path_descarga = kagglehub.dataset_download("mornville/pokemonimagedataset")
print("Ruta de descarga en caché:", path_descarga)

base_dir = Path(path_descarga)

# Búsqueda recursiva para encontrar con exactitud dónde están 'train' y 'test'
rutas_train = list(base_dir.glob("**/train"))
rutas_test = list(base_dir.glob("**/test"))

if not rutas_train:
    raise FileNotFoundError("No se encontró la carpeta 'train' dentro del dataset descargado.")

train_path = rutas_train[0]
test_path = rutas_test[0] if rutas_test else None

print(f"Ruta exacta de Train: {train_path}")
print(f"Ruta exacta de Test:  {test_path}")


## **3. Limpieza Preventiva de Archivos**
Eliminamos archivos que no sean imágenes estándar (como gifs animados aislados) para evitar advertencias o fallos durante la decodificación en TensorFlow.


In [ ]:
extensiones_validas = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def limpiar_archivos_no_imagen(directorio):
    eliminados = 0
    for archivo in list(directorio.glob("*/*")):
        if archivo.is_file() and archivo.suffix.lower() not in extensiones_validas:
            archivo.unlink()
            eliminados += 1
    return eliminados

eliminados_train = limpiar_archivos_no_imagen(train_path)
eliminados_test = limpiar_archivos_no_imagen(test_path) if test_path else 0
print(f"Archivos no compatibles eliminados de Train: {eliminados_train}")
print(f"Archivos no compatibles eliminados de Test:  {eliminados_test}")


## **4. Exploración de Datos (EDA) y Distribución de Clases**
Analizamos la cantidad real de clases de Pokémon y la distribución de imágenes en entrenamiento y prueba.


In [ ]:
# Obtener los nombres de las clases (carpetas de cada especie de Pokémon)
class_names = sorted([d.name for d in train_path.iterdir() if d.is_dir()])
n_classes = len(class_names)
print(f"Total de clases de Pokémon encontradas: {n_classes}")
print("Muestra de primeras 15 clases:", class_names[:15])

# Conteo de imágenes por clase
conteo_train = [len(list((train_path / c).glob("*.*"))) for c in class_names]
conteo_test = [len(list((test_path / c).glob("*.*"))) for c in class_names] if test_path else [0] * n_classes

df_balance = pd.DataFrame({
    "Pokemon": class_names,
    "Train": conteo_train,
    "Test": conteo_test,
    "Total": np.array(conteo_train) + np.array(conteo_test)
})

print(f"Total imágenes en Train: {df_balance['Train'].sum()}")
print(f"Total imágenes en Test:  {df_balance['Test'].sum()}")
print(f"Total de imágenes:       {df_balance['Total'].sum()}")

display(df_balance.describe().round(1))

# Gráfico de barras de una muestra de 25 clases
plt.figure(figsize=(16, 4))
sns.barplot(data=df_balance.head(25), x="Pokemon", y="Train", color="steelblue")
plt.title(f"Distribución de Imágenes en Train (Muestra de 25 de {n_classes} Pokémon)")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## **5. Carga del Dataset y Partición de Validación**
Cargamos las imágenes a 224x224 píxeles. De la carpeta `train` reservamos un 20% para validación interna mediante `validation_split=0.2`, conservando la carpeta `test` para la evaluación final.


In [ ]:
batch_size = 32
img_height = 224
img_width = 224
IMG_SIZE = (img_height, img_width)

# Creación de datasets de entrenamiento y validación
train_dataset, val_dataset = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="both",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=batch_size,
    shuffle=True
)

# Dataset de prueba independiente (sin mezclar)
if test_path:
    test_dataset = tf.keras.utils.image_dataset_from_directory(
        test_path,
        image_size=IMG_SIZE,
        batch_size=batch_size,
        shuffle=False
    )
else:
    val_batches = tf.data.experimental.cardinality(val_dataset)
    test_dataset = val_dataset.take(val_batches // 2)
    val_dataset = val_dataset.skip(val_batches // 2)

print(f"Lotes de Train:      {tf.data.experimental.cardinality(train_dataset)}")
print(f"Lotes de Validación: {tf.data.experimental.cardinality(val_dataset)}")
print(f"Lotes de Test:       {tf.data.experimental.cardinality(test_dataset)}")


## **6. Visualización de Muestras**
Visualizamos un lote de 9 Pokémon del conjunto de entrenamiento junto a sus nombres reales de especie.


In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_dataset.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]], fontsize=10)
        plt.axis("off")
plt.tight_layout()
plt.show()


## **7. Optimización de Carga con AUTOTUNE**
Aplicamos `prefetch` para que la lectura de imágenes desde disco se realice en paralelo con el entrenamiento en GPU.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)


## **8. Aumento de Datos y Preprocesamiento de MobileNetV2**
Definimos capas de aumento de datos (`RandomFlip`, `RandomRotation`, `RandomZoom`) para prevenir sobreajuste y usamos la función oficial `preprocess_input` de MobileNetV2 para normalizar las imágenes a `[-1, 1]`.


In [ ]:
# Aumento de datos (activo solo durante entrenamiento)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.1),
], name="data_augmentation")

# Preprocesamiento específico para MobileNetV2
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input


## **9. Arquitectura MobileNetV2 para Transfer Learning**
Instanciamos MobileNetV2 preentrenada en ImageNet sin la capa de clasificación superior (`include_top=False`).

Congelamos inicialmente toda la base (`base_model.trainable = False`) y agregamos la cabeza clasificadora adecuada para las **{n_classes} clases** de Pokémon.


In [ ]:
IMG_SHAPE = IMG_SIZE + (3,)

# 1. Base convolucional preentrenada
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SHAPE,
    include_top=False,
    weights='imagenet'
)

# 2. Congelar base convolucional
base_model.trainable = False

# 3. Construcción del modelo funcional
inputs = tf.keras.Input(shape=IMG_SHAPE, name="input_image")
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)  # Mantener BatchNormalization en modo inferencia
x = layers.GlobalAveragePooling2D(name="gap")(x)
x = layers.Dropout(0.2, name="dropout")(x)
outputs = layers.Dense(n_classes, activation="softmax", name="predictions")(x)

model = tf.keras.Model(inputs, outputs, name="Pokemon_TransferLearning_MobileNetV2")

# 4. Compilación del modelo (Fase 1)
base_learning_rate = 0.001
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=base_learning_rate),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

model.summary()
print(f"Total de clases en la capa Dense de salida: {n_classes}")
print("Variables entrenables en Fase 1:", len(model.trainable_variables))


## **10. Fase 1: Extracción de Características (*Feature Extraction*)**
Entrenamos únicamente el clasificador denso superior con `EarlyStopping` sobre `val_loss`.


In [ ]:
initial_epochs = 10

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    train_dataset,
    epochs=initial_epochs,
    validation_data=val_dataset,
    callbacks=[early_stopping],
    verbose=1
)


## **11. Curvas de Aprendizaje - Fase 1**
Visualizamos el progreso de Accuracy y Loss durante la primera fase.


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(9, 7))
plt.subplot(2, 1, 1)
plt.plot(acc, label='Train Accuracy')
plt.plot(val_acc, label='Val Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.title('Fase 1: Feature Extraction - Accuracy')
plt.grid(alpha=0.3)

plt.subplot(2, 1, 2)
plt.plot(loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.legend(loc='upper right')
plt.ylabel('Loss (Cross Entropy)')
plt.xlabel('Época')
plt.title('Fase 1: Feature Extraction - Loss')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## **12. Fase 2: Ajuste Fino (*Fine-Tuning*)**
Descongelamos selectivamente las capas convolucionales superiores a partir de la capa 100 y recompilamos con una tasa de aprendizaje reducida ($10^{-4}$) para no perder los pesos preentrenados.


In [ ]:
# Descongelar el modelo base
base_model.trainable = True

print(f"Total de capas en el modelo base: {len(base_model.layers)}")

# Congelar capas anteriores a la capa 100
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Recompilar con learning rate reducido
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(learning_rate=base_learning_rate / 10),
    metrics=['accuracy']
)

model.summary()
print("Variables entrenables en Fase 2 (Fine-Tuning):", len(model.trainable_variables))


In [ ]:
fine_tune_epochs = 10
total_epochs = len(history.epoch) + fine_tune_epochs

history_fine = model.fit(
    train_dataset,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1] + 1,
    validation_data=val_dataset,
    callbacks=[early_stopping],
    verbose=1
)


## **13. Curvas de Aprendizaje Completas**
Generamos la gráfica combinada de Accuracy y Loss con la línea vertical de inicio de Fine-Tuning.


In [ ]:
acc += history_fine.history['accuracy']
val_acc += history_fine.history['val_accuracy']
loss += history_fine.history['loss']
val_loss += history_fine.history['val_loss']

plt.figure(figsize=(10, 8))
plt.subplot(2, 1, 1)
plt.plot(acc, label='Train Accuracy')
plt.plot(val_acc, label='Val Accuracy')
plt.axvline(x=len(history.epoch) - 1, color='green', linestyle='--', label='Inicio Fine-Tuning')
plt.legend(loc='lower right')
plt.title(f'Curvas de Accuracy ({n_classes} clases)')
plt.ylabel('Accuracy')
plt.grid(alpha=0.3)

plt.subplot(2, 1, 2)
plt.plot(loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.axvline(x=len(history.epoch) - 1, color='green', linestyle='--', label='Inicio Fine-Tuning')
plt.legend(loc='upper right')
plt.title('Curvas de Loss (Cross Entropy)')
plt.ylabel('Loss')
plt.xlabel('Época')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('grafica_acc_loss_pokemon.png', dpi=300)
plt.show()


## **14. Evaluación Final en Test y Reporte de Clasificación**
Evaluamos el modelo final sobre las más de 7,000 imágenes de prueba independientes.


In [ ]:
# Evaluación en test
test_loss, test_acc = model.evaluate(test_dataset, verbose=1)
print(f"\nResultados en Test -> Loss: {test_loss:.4f} | Accuracy: {test_acc:.4f} ({test_acc * 100:.2f}%)")

# Predicciones
y_true = np.concatenate([y.numpy() for _, y in test_dataset])
y_probs = model.predict(test_dataset, verbose=1)
y_pred = np.argmax(y_probs, axis=1)

print("\n--- Reporte de Clasificación General ---")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0))


## **15. Matriz de Confusión**
Generamos el mapa de calor de la matriz de confusión multiclase para analizar las predicciones globales.


In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, cmap='Blues', cbar=True, xticklabels=False, yticklabels=False)
plt.title(f'Matriz de Confusión - Test ({n_classes} clases) | Test Accuracy: {test_acc:.3f}')
plt.xlabel('Clase Predicha')
plt.ylabel('Clase Real')
plt.tight_layout()
plt.savefig('matriz_confusion_pokemon.png', dpi=300)
plt.show()

aciertos = int(np.diag(cm).sum())
errores = int(cm.sum() - aciertos)
print(f"Muestras totales de Test: {len(y_true)}")
print(f"Aciertos: {aciertos} ({100 * aciertos / len(y_true):.2f}%)")
print(f"Errores:  {errores} ({100 * errores / len(y_true):.2f}%)")


## **16. Muestras Mal Clasificadas**
Inspeccionamos visualmente los casos en que el modelo cometió un error para entender la confusión (por ejemplo entre evoluciones o ilustraciones complejas).


In [ ]:
mal = np.where(y_pred != y_true)[0]
print(f"Total de imágenes mal clasificadas en test: {len(mal)}")

if len(mal) > 0:
    n_mostrar = min(len(mal), 12)
    columnas = 4
    filas = int(np.ceil(n_mostrar / columnas))
    
    # Extraer imágenes del test dataset
    imagenes_acumuladas = []
    for imgs_batch, _ in test_dataset:
        for img in imgs_batch.numpy():
            imagenes_acumuladas.append(img)
            if len(imagenes_acumuladas) > max(mal[:n_mostrar]):
                break
        if len(imagenes_acumuladas) > max(mal[:n_mostrar]):
            break
            
    fig, axes = plt.subplots(filas, columnas, figsize=(14, 3.5 * filas), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
        
    for ax, idx in zip(axes.ravel(), mal[:n_mostrar]):
        if idx < len(imagenes_acumuladas):
            ax.imshow(imagenes_acumuladas[idx].astype('uint8'))
            real_name = class_names[y_true[idx]]
            pred_name = class_names[y_pred[idx]]
            conf = y_probs[idx].max()
            ax.set_title(f"Real: {real_name}\nPred: {pred_name} ({conf:.0%})", fontsize=9, color='darkred')
    plt.tight_layout()
    plt.show()


## **17. Exportación para Netlify y Aplicación Móvil (TensorFlow.js)**
`tensorflowjs` todavía usa alias que NumPy 2.x eliminó (`np.object`, etc.), y su conversor de modelos Keras no es compatible con Keras 3. Por eso:
1. Aplicamos un parche de compatibilidad a NumPy **antes** de importar `tensorflowjs`.
2. Exportamos el modelo como **SavedModel** (`model.export`) y lo convertimos a **TF.js GraphModel** desde Python (no por CLI, porque el parche no aplica a otro proceso).
3. Resultado en `modelo_web/`: `model.json` + fragmentos `group1-shardXofY.bin` + `class_names.json`.

> En la app web se carga con `tf.loadGraphModel('model.json')`. La entrada es un tensor `[1, 224, 224, 3]` con píxeles en rango 0–255 (el preprocesamiento de MobileNetV2 ya está dentro del modelo).

In [ ]:
import os, json, shutil
import numpy as np
import tensorflow as tf

CARPETA_EXPORT = "modelo_web"
shutil.rmtree(CARPETA_EXPORT, ignore_errors=True)
os.makedirs(CARPETA_EXPORT, exist_ok=True)

# 1) Cargar o verificar el modelo entrenado
ruta_keras = "pokemon_model.keras"
if "model" not in globals():
    print("Cargando modelo desde:", ruta_keras)
    model = tf.keras.models.load_model(ruta_keras)
else:
    model.save(ruta_keras)
    print("Modelo Keras verificado en:", ruta_keras)

# 2) Construir modelo limpio de inferencia (sin RandomFlip/Rotation para producción web/móvil)
# Incluye Rescaling para que acepte directamente píxeles de cámara [0, 255]
try:
    base_m = model.get_layer("mobilenetv2_1.00_224") if "mobilenetv2_1.00_224" in [l.name for l in model.layers] else model.layers[3]
    dense_m = model.get_layer("predictions")
    
    infer_inputs = tf.keras.Input(shape=(224, 224, 3), name="input_1")
    x = tf.keras.layers.Rescaling(1./127.5, offset=-1.0, name="rescaling")(infer_inputs)
    x = base_m(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
    infer_outputs = dense_m(x)
    export_model = tf.keras.Model(infer_inputs, infer_outputs, name="Pokemon_Classifier")
    print("Modelo de inferencia optimizado construido exitosamente.")
except Exception as e:
    print("Usando modelo directo para exportación:", e)
    export_model = model

# 3) Exportación nativa a TensorFlow.js (model.json + group1-shard1of1.bin) sin dependencias rotas
weights_manifest_entries = []
binary_data = bytearray()

for w in export_model.weights:
    w_name = w.name.rstrip(":0")
    w_arr = w.numpy().astype(np.float32)
    weights_manifest_entries.append({
        "name": w_name,
        "shape": list(w_arr.shape),
        "dtype": "float32"
    })
    binary_data.extend(w_arr.tobytes())

bin_filename = "group1-shard1of1.bin"
with open(os.path.join(CARPETA_EXPORT, bin_filename), "wb") as f:
    f.write(binary_data)

# Configuración del topology para TensorFlow.js
try:
    topo = json.loads(export_model.to_json())
except Exception:
    topo = {"class_name": "Functional", "config": export_model.get_config()}

model_json = {
    "format": "layers-model",
    "generatedBy": f"keras v{getattr(tf.keras, '__version__', '3.x')}",
    "convertedBy": "Native Keras-to-TFJS Exporter",
    "modelTopology": topo,
    "weightsManifest": [
        {
            "paths": [bin_filename],
            "weights": weights_manifest_entries
        }
    ]
}

with open(os.path.join(CARPETA_EXPORT, "model.json"), "w", encoding="utf-8") as f:
    json.dump(model_json, f, indent=2)

# 4) Exportar también en formato TFLite (alta velocidad para Android/iOS/WebAssembly)
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(export_model)
    tflite_bytes = converter.convert()
    with open(os.path.join(CARPETA_EXPORT, "model.tflite"), "wb") as f:
        f.write(tflite_bytes)
    print("Formato TFLite (.tflite) generado con éxito.")
except Exception as e:
    print("Nota sobre TFLite:", e)

# 5) Guardar la lista de clases en JSON
with open(os.path.join(CARPETA_EXPORT, "class_names.json"), "w", encoding="utf-8") as f:
    json.dump(class_names, f, indent=2)
print(f"Lista de {len(class_names)} clases guardada en class_names.json")

# 6) Resumen de archivos generados
print(f"\nArchivos generados en {CARPETA_EXPORT}/:")
for arch in sorted(os.listdir(CARPETA_EXPORT)):
    tam_kb = os.path.getsize(os.path.join(CARPETA_EXPORT, arch)) / 1024
    print(f"  - {arch} ({tam_kb:.1f} KB)")

# 7) Comprimir en archivo ZIP y descargar
shutil.make_archive("modelo_web_pokemon", "zip", CARPETA_EXPORT)
print("\n¡ZIP generado con éxito!: modelo_web_pokemon.zip")
try:
    from google.colab import files
    files.download("modelo_web_pokemon.zip")
except ImportError:
    pass

In [ ]:
tiempo_total_mins = (time.time() - tiempo_inicial) / 60
print(f"Tiempo total de ejecución del notebook: {tiempo_total_mins:.2f} minutos")
